**This notebook is a condensed demo of the Venus Cloud Tracker pipeline.**

**It scans the raw NetCDF frames in `data/raw`, finds the first consecutive pair where a cloud feature is successfully detected and tracked, and renders a single velocimetry image showing the tracked feature and its motion vector directly in this notebook (no files written to disk).**

In [ ]:
#Imports

import sys
import os
import pandas as pd
import matplotlib.pyplot as plt

# Add parent directory to path to import venus_tracker
sys.path.append("..")

from venus_tracker.data import load_radiance_datasets
from venus_tracker.processing import get_auto_bounds, final_median, calculate_feature_metrics
from venus_tracker.pipeline import execute_tracking_pipeline, get_evolution_deltas
from venus_tracker.visualization import plot_corrected_velocimetry

print("Modules successfully imported!")

In [ ]:
#Find the first consecutive frame pair with a verified tracked feature

# 1. Define data directory and load pre-downloaded NetCDF datasets
RAW_DATA_DIR = os.path.join("..", "data", "raw")
datasets = load_radiance_datasets(RAW_DATA_DIR)

res1, res2 = None, None
df1, df2 = None, None
df_final = pd.DataFrame()
t1_dt, t2_dt = None, None

print("Scanning loaded datasets for a pair with a verified tracked feature...")

# Loop through consecutive pairs until one produces a fully verified motion vector
for i in range(len(datasets) - 1):
    candidate_1, candidate_2 = datasets[i], datasets[i + 1]

    bounds1 = get_auto_bounds(candidate_1)
    bounds2 = get_auto_bounds(candidate_2)
    if bounds1 is None or bounds2 is None:
        continue

    candidate_res1 = final_median(candidate_1, bounds1['lat_min'], bounds1['lat_max'], bounds1['lon_min'], bounds1['lon_max'])
    candidate_res2 = final_median(candidate_2, bounds2['lat_min'], bounds2['lat_max'], bounds2['lon_min'], bounds2['lon_max'])
    if candidate_res1 is None or candidate_res2 is None:
        continue

    candidate_df1 = calculate_feature_metrics(candidate_res1)
    candidate_df2 = calculate_feature_metrics(candidate_res2)
    if candidate_df1.empty or candidate_df2.empty:
        continue

    candidate_t1_dt = pd.to_datetime(candidate_1.time.values)[0]
    candidate_t2_dt = pd.to_datetime(candidate_2.time.values)[0]
    dt_seconds = (candidate_t2_dt - candidate_t1_dt).total_seconds()

    candidate_results = execute_tracking_pipeline(candidate_res1, candidate_res2, candidate_df1, candidate_df2, dt_seconds)
    if candidate_results.empty:
        continue

    # Confirm at least one candidate vector survives the area/distance consistency checks
    df_evo = candidate_df1.merge(candidate_results, on='feature_id')
    candidate_df_final, _, _ = get_evolution_deltas(df_evo, candidate_df2, next_global_id=1)

    if not candidate_df_final.empty:
        res1, res2 = candidate_res1, candidate_res2
        df1, df2 = candidate_df1, candidate_df2
        df_final = candidate_df_final
        t1_dt, t2_dt = candidate_t1_dt, candidate_t2_dt
        print(f"\n✓ Tracked feature found at index [{i} -> {i+1}]: {t1_dt} -> {t2_dt}")
        break

if res1 is None:
    print("No consecutive pair in data/raw/ produced a verified tracked feature.")


In [ ]:
#Resolve tracked feature IDs and compute velocity/morphology deltas

print(f"Successfully verified {len(df_final)} motion vector(s) out of {len(df1)} candidate feature(s).")
display(df_final.head())


In [ ]:
#Render the tracked feature inline (kept in the notebook only, nothing saved to disk)

t_str = f"{t1_dt.strftime('%Y-%m-%d %H:%M')} -> {t2_dt.strftime('%H:%M')}"

if not df_final.empty:
    plot_corrected_velocimetry(res1, res2, df_final, t_str, save_dir=None)
    plt.show()
else:
    print("Skipping visualization: zero verified vectors for this pair.")
